In [1]:
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python311.zip')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11/lib-dynload')
sys.path.append('/home/amunif/.local/lib/python3.11/site-packages')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11/site-packages')

In [2]:
from sklearn.datasets import make_classification
import numpy as np

import xgboost as xgb

# Using Numpy matrix

<img src="img/XGBRanker%20data%20setup.png" alt="XGBRanker data setup" width="250"/>

In [3]:
# Make a synthetic ranking dataset for demonstration
seed = 1994
X, y = make_classification(random_state=seed)
rng = np.random.default_rng(seed)
n_query_groups = 3
qid = rng.integers(0, n_query_groups, size=X.shape[0])

In [4]:
X

array([[ 1.37274475,  0.55556022, -0.39472313, ..., -1.52678928,
        -0.95222834, -0.58304118],
       [ 0.29797914,  0.60418421,  0.24888156, ..., -0.68180427,
        -0.81937699, -1.8177727 ],
       [-0.20681544,  0.18770098, -0.31246672, ..., -1.71367001,
        -1.77090831, -0.37479097],
       ...,
       [-0.57437761, -1.66850427,  1.18792669, ..., -0.31096033,
         0.80405855,  0.25831555],
       [ 0.86229379,  0.61644239,  1.7479823 , ..., -0.23655313,
        -1.26747288, -0.20751249],
       [-0.34736961, -0.81172299, -0.21660061, ...,  0.05799518,
        -0.45621093,  0.36914458]])

In [5]:
# Sort the inputs based on query index
sorted_idx = np.argsort(qid)
X = X[sorted_idx, :]
y = y[sorted_idx]
qid = qid[sorted_idx]

In [6]:
ranker = xgb.XGBRanker(
            tree_method="hist", 
            lambdarank_num_pair_per_sample=8, 
            objective="rank:pairwise", 
            lambdarank_pair_method="topk"
        )

ranker.fit(X, y, qid=qid)

XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=None, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None,
          lambdarank_num_pair_per_sample=8, lambdarank_pair_method='topk',
          learning_rate=None, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
          max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=None,
          n_jobs=None, ...)

In [7]:
y

array([1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
       1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0,
       1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1,
       0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0])

In [8]:
y_pred = ranker.predict(X)

In [9]:
y_pred[:25]

array([ 2.9092972, -2.9665182, -1.426409 ,  2.4873185, -2.9665182,
        2.9092972, -2.7951186,  2.5976448, -1.8829073, -2.8992999,
       -2.4876425,  1.4951377, -2.8992999,  2.616943 , -2.5652082,
       -2.9665182, -2.9665182, -2.2842166, -2.5652082,  2.545458 ,
        2.391358 , -2.393364 ,  2.5063312, -2.8992999,  2.2269957],
      dtype=float32)

In [10]:
y[:25]

array([1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
       1, 0, 1])

In [11]:
qid[:25]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0])

# Using Pandas Dataframe

In [12]:
import pandas as pd

# `X`, `qid`, and `y` are from the previous snippet, they are all sorted by the `sorted_idx`.
df = pd.DataFrame(X, columns=[str(i) for i in range(X.shape[1])])
df["qid"] = qid

In [13]:
df

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,qid
0,-0.438941,-0.518367,-0.414652,-1.031209,-0.432312,1.252436,0.024844,0.539195,0.270635,0.303833,...,-0.073392,-0.855536,-0.975249,1.240300,0.063219,-1.498230,0.180950,1.129922,-1.040233,0
1,0.198131,-3.074477,0.933927,1.157325,-2.341358,1.617548,-0.687859,1.792938,-0.541408,0.931233,...,-1.000022,-0.079594,-0.828799,-1.961206,-0.821706,2.054504,0.250813,-0.225271,-0.120870,0
2,-0.621530,0.732516,0.269019,0.559307,0.539464,-1.104321,0.941721,-1.897793,1.837516,0.907793,...,0.201258,-0.109952,0.172811,0.331973,1.734853,0.154179,-0.339768,-0.495313,0.185678,0
3,0.388465,1.537994,-2.087239,1.367650,0.377382,2.299039,-1.719528,-0.759369,1.483885,1.389475,...,0.783517,-0.558472,-2.253144,1.688623,1.633251,-0.197652,0.589251,-0.647990,2.221670,0
4,-0.066125,-0.926101,1.407887,-0.773517,0.220194,0.474155,1.121773,-1.250719,-0.252777,0.360188,...,0.909320,-0.399782,-0.103039,-0.908392,0.481251,0.081209,1.068396,0.725095,-0.406591,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.259852,1.328506,-0.180367,-1.313906,0.655758,0.733801,0.117550,1.280856,-0.355781,-1.028894,...,-0.150732,0.964098,2.221375,0.814838,-0.508449,-1.407293,1.080380,-1.165974,0.622051,2
96,-0.206068,0.527394,0.785063,2.222237,0.780396,0.505323,-1.813242,-0.213693,-0.188778,-1.282348,...,0.182024,-0.604494,0.921076,1.735527,-0.151414,0.339606,0.310410,0.326105,0.706655,2
97,0.815276,-0.668167,1.709072,0.949178,1.679298,0.413833,-1.045914,-0.770726,-0.303531,1.084170,...,-0.862140,0.068191,-0.182288,-0.202916,-1.298151,0.763849,0.338292,-0.469184,-1.510361,2
98,0.216709,-0.463331,-0.567728,-1.066087,-1.569823,0.696845,-0.300350,-0.814877,-2.053656,0.423735,...,0.412261,0.476734,0.379004,0.173557,-0.306005,-0.822311,0.990163,-0.393896,0.340422,2


In [14]:
y

array([1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
       1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0,
       1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1,
       0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0])

In [15]:
ranker.fit(df, y)  # No need to pass qid as a separate argument

XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=None, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None,
          lambdarank_num_pair_per_sample=8, lambdarank_pair_method='topk',
          learning_rate=None, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
          max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=None,
          n_jobs=None, ...)

In [16]:
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
# Works with cv in scikit-learn, along with HPO utilities like GridSearchCV
kfold = StratifiedGroupKFold(shuffle=False)
cross_val_score(ranker, df, y, cv=kfold, groups=df.qid)

/group/pmc021/amunif/env/pytorch/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [11:14:17] WARNING: /workspace/src/common/error_msg.cc:52: Empty dataset at worker: 0
  warnings.warn(smsg, UserWarning)


array([1., 1., 1., 0., 0.])

In [17]:
scores = ranker.predict(X)

In [18]:
scores

array([ 2.9092972 , -2.9665182 , -1.426409  ,  2.4873185 , -2.9665182 ,
        2.9092972 , -2.7951186 ,  2.5976448 , -1.8829073 , -2.8992999 ,
       -2.4876425 ,  1.4951377 , -2.8992999 ,  2.616943  , -2.5652082 ,
       -2.9665182 , -2.9665182 , -2.2842166 , -2.5652082 ,  2.545458  ,
        2.391358  , -2.393364  ,  2.5063312 , -2.8992999 ,  2.2269957 ,
       -2.9665182 , -2.5129013 , -2.9665182 , -2.5652082 ,  2.2432833 ,
       -2.8992999 , -2.9665182 , -2.2842166 , -2.9665182 , -2.9665182 ,
        2.9092972 , -2.735018  , -2.8992999 , -2.8992999 , -2.9665182 ,
       -2.9665182 , -2.2842166 ,  2.7986856 , -2.2842166 , -2.9665182 ,
       -2.9665182 , -2.9665182 ,  1.5276957 ,  2.6895382 , -2.9665182 ,
       -2.5652082 ,  2.655516  , -2.9665182 ,  2.9092972 ,  2.7260902 ,
       -2.5129013 , -2.8992999 , -2.9665182 , -2.393364  ,  2.9092972 ,
       -1.7645848 , -2.735018  ,  2.5876126 ,  2.6806126 ,  1.9233576 ,
       -2.5652082 ,  2.9092972 , -2.9665182 ,  2.9092972 ,  2.90

In [19]:
df_full = df

In [20]:
df_full['y'] = y

In [21]:
df_full['score'] = scores

In [22]:
df_full

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,qid,y,score
0,-0.438941,-0.518367,-0.414652,-1.031209,-0.432312,1.252436,0.024844,0.539195,0.270635,0.303833,...,-0.975249,1.240300,0.063219,-1.498230,0.180950,1.129922,-1.040233,0,1,2.909297
1,0.198131,-3.074477,0.933927,1.157325,-2.341358,1.617548,-0.687859,1.792938,-0.541408,0.931233,...,-0.828799,-1.961206,-0.821706,2.054504,0.250813,-0.225271,-0.120870,0,0,-2.966518
2,-0.621530,0.732516,0.269019,0.559307,0.539464,-1.104321,0.941721,-1.897793,1.837516,0.907793,...,0.172811,0.331973,1.734853,0.154179,-0.339768,-0.495313,0.185678,0,1,-1.426409
3,0.388465,1.537994,-2.087239,1.367650,0.377382,2.299039,-1.719528,-0.759369,1.483885,1.389475,...,-2.253144,1.688623,1.633251,-0.197652,0.589251,-0.647990,2.221670,0,1,2.487319
4,-0.066125,-0.926101,1.407887,-0.773517,0.220194,0.474155,1.121773,-1.250719,-0.252777,0.360188,...,-0.103039,-0.908392,0.481251,0.081209,1.068396,0.725095,-0.406591,0,0,-2.966518
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.259852,1.328506,-0.180367,-1.313906,0.655758,0.733801,0.117550,1.280856,-0.355781,-1.028894,...,2.221375,0.814838,-0.508449,-1.407293,1.080380,-1.165974,0.622051,2,1,2.726090
96,-0.206068,0.527394,0.785063,2.222237,0.780396,0.505323,-1.813242,-0.213693,-0.188778,-1.282348,...,0.921076,1.735527,-0.151414,0.339606,0.310410,0.326105,0.706655,2,0,-0.460944
97,0.815276,-0.668167,1.709072,0.949178,1.679298,0.413833,-1.045914,-0.770726,-0.303531,1.084170,...,-0.182288,-0.202916,-1.298151,0.763849,0.338292,-0.469184,-1.510361,2,0,-2.899300
98,0.216709,-0.463331,-0.567728,-1.066087,-1.569823,0.696845,-0.300350,-0.814877,-2.053656,0.423735,...,0.379004,0.173557,-0.306005,-0.822311,0.990163,-0.393896,0.340422,2,1,-1.764585


In [23]:
# df_full.sort_values(by=['qid', 'score'], ascending=[True, False])
df_full.sort_values(by=['score'], ascending=[False])

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,qid,y,score
0,-0.438941,-0.518367,-0.414652,-1.031209,-0.432312,1.252436,0.024844,0.539195,0.270635,0.303833,...,-0.975249,1.240300,0.063219,-1.498230,0.180950,1.129922,-1.040233,0,1,2.909297
68,1.470610,0.331154,-0.431004,-1.168184,-0.096229,0.511850,0.210762,-0.228174,0.156127,-0.013014,...,-0.696226,0.549913,-2.349026,-1.136818,0.767563,-0.844294,0.203013,2,1,2.909297
89,1.573806,-0.395907,-0.235761,-1.295802,-0.224563,-0.323659,-0.446189,-0.846448,2.498920,-0.465098,...,0.019470,1.885331,2.059463,-2.096817,-0.100299,-0.527940,-0.115722,2,1,2.909297
84,-1.792814,0.505197,-0.758400,-0.971713,0.448603,-0.501972,0.948744,1.001042,0.087772,0.089457,...,-0.961131,1.355219,1.864423,-1.533999,0.337681,-0.460447,-2.172854,2,1,2.909297
77,-0.410220,-0.087918,-0.269315,-0.991832,-1.606111,-0.056275,1.737854,1.890486,-0.101579,0.123377,...,-0.957908,1.107039,0.904915,-1.384724,0.456717,-2.231578,-1.713816,2,1,2.909297
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,-0.661567,-1.804844,0.326717,-0.600924,2.861517,-0.655291,0.444601,-0.097848,0.170601,-0.256602,...,0.016688,-0.836024,-0.652587,0.148495,-0.005722,-0.219135,-1.006673,1,0,-2.966518
70,0.301283,0.480354,-0.890163,0.391742,1.883123,0.902812,1.174342,-2.048452,0.551580,-2.055782,...,1.459908,-1.243328,-0.731737,1.075196,0.369656,0.484538,0.551434,2,0,-2.966518
76,1.888667,0.814382,0.402607,0.407186,0.150238,0.482765,-0.567565,-1.173693,0.254967,0.433210,...,-1.241469,-1.021642,-1.099080,0.940177,-0.759097,-1.060872,0.532731,2,0,-2.966518
31,1.664004,2.125111,-1.671314,0.884688,0.496079,0.307614,-0.731273,0.407653,1.638810,2.821789,...,-0.995864,-0.830841,-0.781196,1.132502,-0.311124,0.819769,-0.618765,1,0,-2.966518


In [24]:
df_full.sort_values(by=['score'], ascending=[False]).to_csv('df_full_ranking.csv', header=True, index=False)

In [25]:
df_full[df_full['qid'] == 0].sort_values(by=['score'], ascending=[False])

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,qid,y,score
0,-0.438941,-0.518367,-0.414652,-1.031209,-0.432312,1.252436,0.024844,0.539195,0.270635,0.303833,...,-0.975249,1.240300,0.063219,-1.498230,0.180950,1.129922,-1.040233,0,1,2.909297
5,-0.723411,-1.238955,-0.639608,-1.354106,-0.341687,1.321977,0.985204,-3.144467,1.477264,0.278815,...,-1.348010,0.367322,0.168023,-1.140728,-2.213505,-0.750020,-1.186801,0,1,2.909297
13,-0.431667,-0.112337,0.563735,1.207654,0.295309,1.308372,0.774756,-0.001372,1.408477,-1.180945,...,-0.763608,1.280704,-0.146758,-0.036660,-1.571850,-1.203611,1.585200,0,1,2.616943
7,2.343144,-2.389513,0.245658,1.251877,-1.128762,0.548448,1.886634,1.014214,0.204521,-0.883561,...,1.050530,1.594310,-1.068800,-0.212792,-0.432662,0.027232,1.808023,0,1,2.597645
19,-1.106894,-1.096703,0.777550,1.297725,-0.756983,1.012394,-0.412962,-0.296868,-1.457080,-0.801765,...,1.046101,1.476697,0.196245,-0.105240,-0.655125,-0.226297,0.702951,0,1,2.545458
22,-0.574378,-1.668504,1.187927,1.198080,0.761331,-0.338965,0.073198,0.760785,0.557793,-1.129361,...,1.343468,1.141622,1.094058,0.048126,-0.310960,0.804059,0.258316,0,1,2.506331
3,0.388465,1.537994,-2.087239,1.367650,0.377382,2.299039,-1.719528,-0.759369,1.483885,1.389475,...,-2.253144,1.688623,1.633251,-0.197652,0.589251,-0.647990,2.221670,0,1,2.487319
20,0.762844,-1.032098,0.623401,1.584440,0.140854,0.473208,1.303675,0.280904,1.072950,0.104081,...,1.957248,2.037466,0.136500,-0.282181,1.359246,-1.271619,0.657244,0,1,2.391358
24,1.069908,0.106951,0.072171,0.658334,0.055424,-0.926297,0.445065,0.566338,1.722302,2.102346,...,-1.038563,0.473346,1.069357,0.127346,0.719949,-1.317175,-0.111606,0,1,2.226996
11,-0.453093,0.233676,-0.421557,1.157399,-0.238253,0.996704,-1.235808,1.182060,1.469812,1.036746,...,-0.707910,0.795445,0.266198,0.247957,0.270318,0.347529,-0.299004,0,1,1.495138


## Sorting the scores

In [26]:
# sorted_idx = np.argsort(scores)[::-1]
# # Sort the relevance scores from most relevant to least relevant
# scores = scores[sorted_idx]

In [27]:
# scores